# SQLite through Python

First steps for reading and modifying your SQLite database using Python (and Pandas).

## SQLite vs. SQL vs. Python vs. pandas

**SQLite** — a file. That's it. It's a little database that lives as a single file on your computer (like `mydata.db`), holding all your tables of data. No server, no setup, just a file you can pass around.

**SQL** — a language. Not a program, not a file — it's the language you use to *talk to* a database. Things like `SELECT * FROM table WHERE...`. Every database engine (SQLite, Postgres, MySQL) speaks some flavor of SQL.

**Python** — a general-purpose programming language. It doesn't know anything about databases on its own, but it can do basically everything else: loops, logic, file handling, web requests, you name it.

**pandas** — a Python library (a bundle of pre-written code you import) that gives you the "DataFrame" — a spreadsheet-like table you can manipulate right inside Python.

### Why nest SQL inside Python?

You *could* open SQLite on its own and type SQL commands directly into it. But instead, we're going to write Python code that contains SQL commands as strings, and Python sends those commands to the SQLite file for us.

Why bother? A couple of reasons:
- It's just easier — you get one script that connects to the database, runs your query, and hands you the results, all in one go.
- It opens doors. Once your data's in Python, you can dump it straight into pandas for analysis, plot it, clean it up, loop over it, combine it with other data sources, automate it to run on a schedule — stuff that's clunky or impossible in raw SQL alone.

Basically: SQLite holds the data, SQL is how you ask for it, and Python + pandas is where you actually *do* things with it once you've got it.

## Importing libraries

In Python, we use `import` to load code from other modules so we can reuse useful tools without rewriting them ourselves. For this notebook, we will import a few standard Python and project libraries:

- `sqlite3` to connect to and work with SQLite databases
- `os` to work with file paths and folders on our computer
- `pandas` to organize query results in tables

This lets us connect to the database, access the file path we need, and keep the notebook organized and readable.

### Installing a library vs. importing it

A very important distinction: installing a library is a one-time setup step, while `import` is something we do inside the code every time we want to use it.

For example, if we need `pandas`, we might first install it once on our computer using a package manager such as Homebrew or `pip`. After that, we do not reinstall it each time we run code. Instead, in the notebook we simply write:

```python
import pandas as pd
```

This tells Python, "use the pandas library that is already installed on this machine." It does not install pandas again; it only tells Python which package to load into this session.

A typical one-time install command looks like this:

```bash
pip install pandas
```

After that, we do not run that command again every time. We simply import the library when we need it in the notebook or script.

The same idea applies to all the libraries we use here. We install them once so they are available on the computer, and then we `import` them when we need them in a script or notebook.

In [2]:
import sqlite3
import os
import pandas as pd

In order to interact with the SQLite database file, we need to tell Python where it lives. The simplest way to do that is to save the file's location as a plain Python string, and then reuse that string later whenever we need to connect to the database. We are not doing anything fancy here — we are just storing a path as text so Python can use it again and again.

A good place to start is your home directory, which you can access with `~` in Python. If you want to build the path manually, you can open Finder, locate the database file, hold down `Option` and right-click on the file, then choose the option to copy its path name. That gives you the exact string you can paste into Python.

This is still just a string variable like any other: `database_path = "/Users/yourname/.../your_database.db"`.

In [ ]:
# Start from your home directory and then add the rest of the file path as a string.
# On a Mac, you can hold down Option and right-click a file in Finder to copy its path.
home_dir = os.path.expanduser("~")
database_path = home_dir + "/Dropbox/Active_Directories/Digital_Humanities/database_eurasia_7.0.db"

# This is just a string variable that stores the database location for later use.
# For your own file, replace the example path with your own path.
# Example:
# database_path = home_dir + "/your_folder/your_subfolder/your_database.db"

## Taking a first look at the database

Now that we know where the SQLite file is, the first thing we usually do is inspect it. We are not querying the actual data yet; we are asking the database for its structure and metadata.

In other words, we want to answer a few basic questions:
- What tables does the database contain?
- What fields are inside each table?
- What are the field types?
- Which columns are unique identifiers or join keys?

This is the “look around and understand the schema” step before we do any real analysis.

A simple SQL question looks like this:

```sql
SELECT name
FROM sqlite_master
WHERE type = 'table';
```

This asks SQLite: show me all the tables in this database.

We can also ask for field information using a schema command:

```sql
PRAGMA table_info(table_name);
```

This returns information about each field in a table, including:
- the column name
- the data type
- whether it is a primary key
- whether it is allowed to be null

In Python, we will use `sqlite3` to connect to the database and run these SQL statements. The Python code will not be doing anything complicated — it is just sending SQL to SQLite and printing the results so we can read them.

In [4]:
# Connect to the SQLite database using the path we stored earlier.
conn = sqlite3.connect(database_path)
cursor = conn.cursor()

# 1) List all tables in the database.
cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;")
tables = cursor.fetchall()
print("Tables in the database:")
for table in tables:
    print("-", table[0])

print("\n")

# 2) Inspect the fields for one table.
# Replace 'lexicon' with any table you want to inspect.
example_table = 'lexicon'
cursor.execute(f"PRAGMA table_info({example_table});")
columns = cursor.fetchall()

print(f"Fields in '{example_table}':")
for col in columns:
    cid, name, data_type, notnull, default_value, pk = col
    print(f"- {name}: {data_type} | primary_key={bool(pk)} | not_null={bool(notnull)}")

cursor.close()
conn.close()

Tables in the database:
- bibliography
- classical_genre
- classical_sources
- commentaries
- conquests
- copies_holdings
- definitions
- epochs
- gazetteer
- honorifics
- individual_social_roles
- itineraries
- knowledge_branch
- knowledge_forms
- knowledge_mastery
- lexicon
- location_attributes
- location_hierarchies
- multiple_sources_conquests
- prices
- prosopography
- references_to_classical_sources
- references_to_individuals
- references_to_locations
- related_sources
- related_terms
- relationships
- repositories
- role_honorific
- seals
- social_roles


Fields in 'lexicon':
- UID: INTEGER | primary_key=True | not_null=False
- Term: TEXT | primary_key=False | not_null=False
- Translation: TEXT | primary_key=False | not_null=False
- Scope: TEXT | primary_key=False | not_null=False
- Colonial_Term: TEXT | primary_key=False | not_null=False
- Emic_Term: TEXT | primary_key=False | not_null=False
- Etymology: TEXT | primary_key=False | not_null=False
- Notes: TEXT | primary_key=Fa

## Writing our own Python functions

A function is a reusable block of code that does one job. It lets us package a few steps into a single named unit, so we can call it again later instead of rewriting the same code every time.

The basic pattern is:

```python
def function_name(parameters):
    # code to do something
    return result
```

- `def` tells Python, "this is a function"
- `function_name` is the name we will use later
- `parameters` are values we pass in, like a table name or a search term
- the body runs the logic
- `return` gives us back the result we want

Why do this here? Because we are going to build a small search function that we can reuse for our own database. It will:
- connect to the SQLite database
- choose a table and field to search
- look for a matching value using SQL
- return the matching rows

This is a simple first version, not a perfect production search tool. The goal is to make it clear and understandable, so we can learn the logic before adding more power later.

## Building a custom search function

The next step is to write a very simple custom search function that matches our own database structure. We do not need to build a huge system yet. We just want a function that takes:
- a table name
- a field name
- a value to search for

and then returns matching rows.

This version will be intentionally basic. It will search one field at a time, use a simple SQL `SELECT` statement, and print the results in a readable way. Later, we can add more complexity, such as multiple search fields, joins, exports, or more flexible matching.

The key idea is that the function is just a wrapper around a normal SQL query. The database is still doing the actual work; Python is simply building the query and handling the results.

Example pattern:

```python
def simple_search(table_name, field_name, search_value):
    conn = sqlite3.connect(database_path)
    cursor = conn.cursor()
    query = f"SELECT * FROM {table_name} WHERE {field_name} = ?"
    cursor.execute(query, (search_value,))
    results = cursor.fetchall()
    conn.close()
    return results
```

This is a good first version because every part is easy to understand:
- `table_name` tells us which table to query
- `field_name` tells us which column to search
- `search_value` is the specific value we want
- `?` is a placeholder for the value, which helps keep the query safe
- `fetchall()` gets all matching rows

This is the kind of function we can build ourselves and then adapt to our own database as we learn more.


In [ ]:
def simple_search(table_name, field_name, search_value):
    """Search a single column in a table for an exact match."""
    conn = sqlite3.connect(database_path)
    cursor = conn.cursor()

    query = f"SELECT * FROM {table_name} WHERE {field_name} = ?"
    cursor.execute(query, (search_value,))
    results = cursor.fetchall()

    cursor.close()
    conn.close()
    return results

# Example use:
# rows = simple_search('lexicon', 'Word', 'example')
# print(rows)

# You can replace 'lexicon' and 'Word' with any table/field names from your database.
# This version is intentionally simple and easy to read.
